# 解析結果ビューア

この notebook は、既存 pipeline が `results/` 配下に保存した CSV、JSON、画像、HTML レポートを横断的に確認するための探索用ビューアです。元 pcap や大規模 flow CSV を再処理せず、summary / comparison / prefix evaluation / 既存 plot を読み込みます。

入力: `results/comparison/<dataset>/comparison_summary.csv`、`results/prefix/<dataset>/*.csv`、`results/features/**/features.json`、`results/comparison/<dataset>/plots/` など。  
出力: notebook 上の表と軽量な確認図のみ。必要に応じて既存画像を表示します。

注意: 短命 flow、RST、scan_candidate は異常と断定せず、prefix の性質を示唆する指標として確認します。

In [ ]:
# 設定セル: まずここだけ変更してください。
DATASET = None  # 例: "202604081300"。None の場合は results/ から自動選択。
TARGET = None   # 例: "dst_157.209.123.224_27" または prefix 文字列。None の場合は候補から自動選択。
TARGET_UNIT = "prefix"
FEATURE_NAMES = [
    "flow_inter_arrival_time",
    "duration",
    "packet_count",
    "byte_count",
    "pps",
    "bps",
    "avg_packet_size",
    "packets_from_src_ratio",
    "bytes_from_src_ratio",
]
TOP_K = 15
IMAGE_MAX_WIDTH = 760
SHOW_EXISTING_PLOTS = True

In [ ]:
from __future__ import annotations

import json
import math
import re
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, Image, display

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["font.size"] = 10
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("project root not found: expected AGENTS.md and results/")


PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / "results"
print(f"PROJECT_ROOT = {PROJECT_ROOT}")


def rel(path: Path | None) -> str:
    if path is None:
        return ""
    try:
        return str(path.resolve().relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


def read_csv(path: Path | None) -> pd.DataFrame:
    if path is None or not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


def read_json(path: Path | None) -> dict[str, Any]:
    if path is None or not path.exists():
        return {}
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    return data if isinstance(data, dict) else {}


def available_datasets() -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    sources = {
        "comparison": RESULTS_DIR / "comparison",
        "prefix": RESULTS_DIR / "prefix",
        "features_all": RESULTS_DIR / "features" / "all",
        "aguri": RESULTS_DIR / "aguri",
        "reports": RESULTS_DIR / "reports",
    }
    names = sorted({p.name for base in sources.values() if base.exists() for p in base.iterdir() if p.is_dir()})
    for name in names:
        rows.append({
            "dataset": name,
            "comparison_summary": (RESULTS_DIR / "comparison" / name / "comparison_summary.csv").exists(),
            "prefix_evaluation": (RESULTS_DIR / "prefix" / name / "prefix_evaluation.csv").exists(),
            "selected_prefixes": (RESULTS_DIR / "prefix" / name / "selected_prefixes.csv").exists(),
            "overall_features": (RESULTS_DIR / "features" / "all" / name / "features.json").exists(),
            "aguri_candidates": (RESULTS_DIR / "aguri" / name / f"{name}.aguri_candidates.csv").exists(),
            "report_html": (RESULTS_DIR / "reports" / name / "analysis_report.html").exists(),
        })
    return pd.DataFrame(rows)


def choose_dataset(configured: str | None) -> str:
    datasets = available_datasets()
    if datasets.empty:
        raise FileNotFoundError("no result datasets found under results/")
    if configured:
        if configured not in set(datasets["dataset"]):
            print(f"[warn] configured dataset not discovered: {configured}")
        return configured
    scored = datasets.assign(score=datasets.drop(columns=["dataset"]).sum(axis=1))
    return str(scored.sort_values(["comparison_summary", "prefix_evaluation", "score", "dataset"], ascending=[False, False, False, True]).iloc[0]["dataset"])


def dataset_paths(dataset: str) -> dict[str, Path]:
    return {
        "comparison_summary": RESULTS_DIR / "comparison" / dataset / "comparison_summary.csv",
        "comparison_plots": RESULTS_DIR / "comparison" / dataset / "plots",
        "prefix_evaluation": RESULTS_DIR / "prefix" / dataset / "prefix_evaluation.csv",
        "selected_prefixes": RESULTS_DIR / "prefix" / dataset / "selected_prefixes.csv",
        "overall_features": RESULTS_DIR / "features" / "all" / dataset / "features.json",
        "flow_plots_all": RESULTS_DIR / "flow_plots" / "all" / dataset,
        "flow_plots_prefix": RESULTS_DIR / "flow_plots" / "prefix" / dataset,
        "aguri_candidates": RESULTS_DIR / "aguri" / dataset / f"{dataset}.aguri_candidates.csv",
        "report_html": RESULTS_DIR / "reports" / dataset / "analysis_report.html",
    }


def file_audit(paths: dict[str, Path]) -> pd.DataFrame:
    rows = []
    for key, path in paths.items():
        if path.is_dir():
            count = len([p for p in path.rglob("*") if p.is_file()])
            rows.append({"name": key, "path": rel(path), "exists": True, "type": "dir", "items": count, "rows": None, "columns": None})
        elif path.exists():
            rows.append({"name": key, "path": rel(path), "exists": True, "type": path.suffix.lstrip("."), "items": None, "rows": safe_row_count(path), "columns": safe_columns(path)})
        else:
            rows.append({"name": key, "path": rel(path), "exists": False, "type": "", "items": None, "rows": None, "columns": None})
    return pd.DataFrame(rows)


def safe_row_count(path: Path) -> int | None:
    if path.suffix.lower() != ".csv":
        return None
    try:
        return len(pd.read_csv(path))
    except Exception:
        return None


def safe_columns(path: Path) -> str | None:
    try:
        if path.suffix.lower() == ".csv":
            return ", ".join(pd.read_csv(path, nrows=0).columns)
        if path.suffix.lower() == ".json":
            return ", ".join(read_json(path).keys())
    except Exception:
        return None
    return None


def missing_summary(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()
    return pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[c].dtype) for c in df.columns],
        "missing": [int(df[c].isna().sum()) for c in df.columns],
        "missing_ratio": [float(df[c].isna().mean()) for c in df.columns],
    })


def first_existing_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    lowered = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in df.columns:
            return cand
        if cand.lower() in lowered:
            return lowered[cand.lower()]
    return None


def numeric_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    col = first_existing_column(df, candidates)
    if col is None:
        return None
    converted = pd.to_numeric(df[col], errors="coerce")
    if converted.notna().any():
        df[col] = converted
        return col
    return None


def target_col(df: pd.DataFrame) -> str | None:
    return first_existing_column(df, ["target", "normalized_dst_prefix", "dst_prefix", "prefix", "aggregate_id"])


def sanitize_target(name: str) -> str:
    return "".join(c if c.isalnum() or c in ("-", "_", ".") else "_" for c in str(name))


def target_candidates(comparison: pd.DataFrame, selected: pd.DataFrame, evaluation: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for source, df in [("comparison_summary", comparison), ("selected_prefixes", selected), ("prefix_evaluation", evaluation)]:
        col = target_col(df)
        if col is None:
            continue
        for _, row in df.iterrows():
            value = str(row[col])
            if value.lower() == "overall":
                continue
            rows.append({"source": source, "target": value, "plot_dir_name": sanitize_target(value), **{k: row[k] for k in df.columns if k in {"flow_count", "packet_count", "byte_count", "score", "scan_candidate", "passes_filters"}}})
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.drop_duplicates(["source", "target"]).reset_index(drop=True)


def choose_target(configured: str | None, candidates: pd.DataFrame) -> str | None:
    if configured:
        return configured
    if candidates.empty:
        return None
    for source in ["selected_prefixes", "comparison_summary", "prefix_evaluation"]:
        sub = candidates[candidates["source"] == source]
        if not sub.empty:
            return str(sub.iloc[0]["target"])
    return str(candidates.iloc[0]["target"])


def find_target_plot_dir(plots_root: Path, target: str | None) -> Path | None:
    if target is None or not plots_root.exists():
        return None
    names = [sanitize_target(target)]
    if "/" in str(target):
        names.append(sanitize_target(str(target).replace("/", "_")))
    existing = {p.name: p for p in plots_root.iterdir() if p.is_dir()}
    for name in names:
        if name in existing:
            return existing[name]
    target_norm = sanitize_target(target).lower()
    for name, path in existing.items():
        if name.lower() == target_norm or target_norm in name.lower() or name.lower() in target_norm:
            return path
    return None


def image_grid(paths: list[Path], max_width: int = 720) -> None:
    if not paths:
        print("no images found")
        return
    for path in paths:
        display(HTML(f"<div style='font-weight:600;margin-top:12px'>{rel(path)}</div>"))
        display(Image(filename=str(path), width=max_width))


def feature_rows(features_json: dict[str, Any]) -> pd.DataFrame:
    rows = []
    for name, block in features_json.get("features", {}).items():
        stats = block.get("stats", {}) if isinstance(block, dict) else {}
        rows.append({"feature": name, "unit": block.get("unit", ""), "log_scale_recommended": block.get("log_scale_recommended", False), **stats})
    return pd.DataFrame(rows)


def histogram_block(features_json: dict[str, Any], feature: str, prefer_log: bool = True) -> tuple[str, dict[str, Any], dict[str, Any]] | None:
    block = features_json.get("features", {}).get(feature)
    if not isinstance(block, dict):
        return None
    if prefer_log and block.get("log_scale_recommended") and isinstance(block.get("log_histogram"), dict):
        return "log_histogram", block["log_histogram"], block
    if isinstance(block.get("histogram"), dict):
        return "histogram", block["histogram"], block
    if isinstance(block.get("log_histogram"), dict):
        return "log_histogram", block["log_histogram"], block
    return None


def hist_edges(hist_type: str, hist: dict[str, Any]) -> list[float] | None:
    raw = hist.get("linear_edges") if hist_type == "log_histogram" else hist.get("edges")
    if raw is None:
        raw = hist.get("bins") if isinstance(hist.get("bins"), list) else None
    if not isinstance(raw, list):
        return None
    try:
        return [float(v) for v in raw]
    except Exception:
        return None


def hist_counts(hist: dict[str, Any]) -> list[float] | None:
    raw = hist.get("counts")
    if not isinstance(raw, list):
        return None
    try:
        return [float(v) for v in raw]
    except Exception:
        return None


def plot_hist_and_cdf(features_json: dict[str, Any], feature_names: list[str], title_prefix: str = "") -> None:
    available = [f for f in feature_names if histogram_block(features_json, f) is not None]
    if not available:
        print("no plottable histogram features found")
        return
    fig, axes = plt.subplots(len(available), 2, figsize=(12, max(3.2, 2.8 * len(available))))
    axes = np.atleast_2d(axes)
    for row, feature in enumerate(available):
        hist_type, hist, block = histogram_block(features_json, feature)
        edges = hist_edges(hist_type, hist)
        counts = hist_counts(hist)
        if edges is None or counts is None or len(edges) != len(counts) + 1:
            continue
        widths = np.diff(edges)
        axes[row, 0].bar(edges[:-1], counts, width=widths, align="edge", color="#4C78A8", edgecolor="white", linewidth=0.4)
        if hist_type == "log_histogram":
            axes[row, 0].set_xscale("log")
        axes[row, 0].set_title(f"{feature} histogram")
        axes[row, 0].set_ylabel("flow count")
        total = np.sum(counts)
        cdf = np.cumsum(counts) / total if total else np.zeros(len(counts))
        axes[row, 1].step(edges[1:], cdf, where="post", color="#F58518")
        if hist_type == "log_histogram":
            axes[row, 1].set_xscale("log")
        axes[row, 1].set_ylim(0, 1.02)
        axes[row, 1].set_title(f"{feature} CDF")
        axes[row, 1].set_ylabel("cumulative ratio")
    if title_prefix:
        fig.suptitle(title_prefix, y=1.01, fontsize=12)
    fig.tight_layout()
    plt.show()


def plot_topk(df: pd.DataFrame, metric: str, label_col: str | None = None, top_k: int = 15, title: str = "") -> None:
    if df.empty or metric not in df.columns:
        print(f"skip top-k: missing {metric}")
        return
    label_col = label_col or target_col(df)
    if label_col is None:
        print("skip top-k: no label column")
        return
    work = df.copy()
    work[metric] = pd.to_numeric(work[metric], errors="coerce")
    work = work.dropna(subset=[metric]).sort_values(metric, ascending=False).head(top_k)
    if work.empty:
        print(f"skip top-k: no numeric values for {metric}")
        return
    fig, ax = plt.subplots(figsize=(9, max(3.0, 0.36 * len(work) + 1.0)))
    ax.barh(work[label_col].astype(str)[::-1], work[metric][::-1], color="#4C78A8", edgecolor="white")
    ax.set_xlabel(metric)
    ax.set_title(title or f"Top {len(work)} by {metric}")
    fig.tight_layout()
    plt.show()

In [ ]:
datasets = available_datasets()
display(datasets)

dataset = choose_dataset(DATASET)
paths = dataset_paths(dataset)
print(f"selected dataset = {dataset}")
display(file_audit(paths))

## 入力ファイルの確認

読み込んだ CSV の行数、列名、欠損を先に確認します。対象期間は `features.json` の `totals` に保存されている場合に表示します。

In [ ]:
comparison = read_csv(paths["comparison_summary"])
prefix_eval = read_csv(paths["prefix_evaluation"])
selected = read_csv(paths["selected_prefixes"])
aguri = read_csv(paths["aguri_candidates"])
overall_features = read_json(paths["overall_features"])

for name, df in [("comparison_summary", comparison), ("prefix_evaluation", prefix_eval), ("selected_prefixes", selected), ("aguri_candidates", aguri)]:
    print(f"\n{name}: rows={len(df)}, columns={list(df.columns)}")
    if not df.empty:
        display(missing_summary(df).head(80))

totals = overall_features.get("totals", {})
if totals:
    display(pd.DataFrame([{
        "features_json": rel(paths["overall_features"]),
        "valid_flow_count": totals.get("valid_flow_count"),
        "packet_total": totals.get("packet_total"),
        "byte_total": totals.get("byte_total"),
        "first_start_time": totals.get("first_start_time"),
        "last_end_time": totals.get("last_end_time"),
        "capture_span_seconds": totals.get("capture_span_seconds"),
    }]))

## 対象一覧

`comparison_summary`、`selected_prefixes`、`prefix_evaluation` に現れる対象を一覧化します。`TARGET` が未設定の場合は、選定済み prefix、比較 summary、評価結果の順に先頭候補を選びます。

In [ ]:
candidates = target_candidates(comparison, selected, prefix_eval)
display(candidates.head(100))

target = choose_target(TARGET, candidates)
print(f"selected target = {target}")
plot_dir = find_target_plot_dir(paths["comparison_plots"], target)
print(f"target plot dir = {rel(plot_dir) if plot_dir else None}")

## Summary table と ranking

prefix 選定では通信量だけでなく、flow 数、短命 flow、tiny flow、RST/SYN-only-like、TCP/UDP 比率、MAWI 全体との差を合わせて確認します。

In [ ]:
for name, df in [("comparison_summary", comparison), ("selected_prefixes", selected), ("prefix_evaluation", prefix_eval), ("aguri_candidates", aguri)]:
    if df.empty:
        continue
    print(f"\n{name}")
    display(df.head(30))

ranking_specs = [
    (prefix_eval, "score"),
    (prefix_eval, "flow_count"),
    (prefix_eval, "byte_count"),
    (prefix_eval, "short_flow_ratio"),
    (prefix_eval, "tiny_flow_ratio"),
    (comparison, "flow_count"),
    (comparison, "duration_mean"),
    (comparison, "byte_count_mean"),
]
for df, metric in ranking_specs:
    if not df.empty and metric in df.columns:
        plot_topk(df[df.get("target", pd.Series([""] * len(df))).astype(str).str.lower() != "overall"] if "target" in df.columns else df, metric, top_k=TOP_K)

## 対象の既存 plot

比較 pipeline が生成済みの画像を表示します。ここでは再作図ではなく、`results/comparison/<dataset>/plots/<target>/` の既存 plot を確認します。

In [ ]:
if SHOW_EXISTING_PLOTS and plot_dir:
    images = sorted([p for p in plot_dir.glob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".pdf"}])
    image_grid(images, max_width=IMAGE_MAX_WIDTH)
else:
    print("existing plot display disabled or plot directory not found")

## features.json からの横断確認

`features.json` に保存済みの histogram / CDF を使って、複数特徴量を同じ notebook 上で確認します。flow CSV からの再集計は行いません。

In [ ]:
if overall_features:
    display(feature_rows(overall_features))
    plot_hist_and_cdf(overall_features, FEATURE_NAMES, title_prefix=f"overall: {dataset}")
else:
    print("overall features.json not found for this dataset")

prefix_feature_root = RESULTS_DIR / "features" / "prefix" / dataset
prefix_jsons = sorted(prefix_feature_root.glob("*_features.json")) if prefix_feature_root.exists() else []
print(f"prefix features found: {len(prefix_jsons)} under {rel(prefix_feature_root)}")
if prefix_jsons:
    selected_prefix_json = prefix_jsons[0]
    if target:
        target_norm = sanitize_target(target).lower()
        for candidate in prefix_jsons:
            if target_norm in candidate.stem.lower() or candidate.stem.lower() in target_norm:
                selected_prefix_json = candidate
                break
    print(f"selected prefix features = {rel(selected_prefix_json)}")
    prefix_features = read_json(selected_prefix_json)
    display(feature_rows(prefix_features))
    plot_hist_and_cdf(prefix_features, FEATURE_NAMES, title_prefix=selected_prefix_json.stem)

## 解釈メモ用の確認セル

このセルは、対象 prefix がどの指標で MAWI 全体と異なるかを確認するための軽い差分表です。差分は探索の入口であり、単独で攻撃・正常を断定するものではありません。

In [ ]:
if not comparison.empty and target:
    tcol = target_col(comparison)
    if tcol:
        overall_row = comparison[comparison[tcol].astype(str).str.lower() == "overall"]
        target_rows = comparison[(comparison[tcol].astype(str) == str(target)) | (comparison[tcol].astype(str) == sanitize_target(str(target)))]
        if target_rows.empty and plot_dir is not None:
            target_rows = comparison[comparison[tcol].astype(str) == plot_dir.name]
        if not overall_row.empty and not target_rows.empty:
            num_cols = comparison.select_dtypes(include="number").columns
            diff = pd.DataFrame({
                "metric": num_cols,
                "overall": [overall_row.iloc[0][c] for c in num_cols],
                "target": [target_rows.iloc[0][c] for c in num_cols],
            })
            diff["target_minus_overall"] = diff["target"] - diff["overall"]
            diff["target_over_overall"] = diff.apply(lambda r: r["target"] / r["overall"] if r["overall"] not in [0, None] and pd.notna(r["overall"]) else np.nan, axis=1)
            display(diff.sort_values("target_minus_overall", key=lambda s: s.abs(), ascending=False))
        else:
            print("comparison rows for overall/target were not found")